In [1]:
import requests
import pandas as pd
import time


# ============================================================
# SETTINGS
# ============================================================

# TheSportsDB's public/free V1 API key
API_KEY = "123"

BASE_URL = f"https://www.thesportsdb.com/api/v1/json/{API_KEY}"

# NBA season we want
SEASON = "2025-2026"


# ============================================================
# HELPER FUNCTION
# ============================================================

def make_request(endpoint, params=None):
    """
    Sends a GET request to TheSportsDB API
    and returns JSON data.
    """

    url = f"{BASE_URL}/{endpoint}"

    print(f"\nRequesting: {url}")

    try:
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        # Stop if request fails
        response.raise_for_status()

        return response.json()

    except requests.exceptions.RequestException as error:
        print("API request failed:")
        print(error)

        return None


# ============================================================
# 1. FIND THE NBA LEAGUE
# ============================================================

def find_nba_league():
    """
    Searches TheSportsDB for basketball leagues
    in the United States and finds the NBA.
    """

    print("\n====================================")
    print("STEP 1: FINDING NBA LEAGUE")
    print("====================================")

    data = make_request(
        "search_all_leagues.php",
        {
            "c": "United States",
            "s": "Basketball"
        }
    )

    if data is None:
        return None

    leagues = data.get("countries")

    if not leagues:
        print("No basketball leagues were returned.")
        return None

    for league in leagues:

        league_name = league.get("strLeague", "")

        print("Found league:", league_name)

        if league_name.upper() == "NBA":
            print("\nNBA FOUND!")
            print("League ID:", league.get("idLeague"))

            return league

    # If exact NBA was not found, search for anything
    # containing NBA
    for league in leagues:

        league_name = league.get("strLeague", "")

        if "NBA" in league_name.upper():

            print("\nNBA-like league found!")
            print("League:", league_name)
            print("League ID:", league.get("idLeague"))

            return league

    print("\nCould not automatically find NBA.")

    return None


# ============================================================
# 2. GET NBA TEAMS
# ============================================================

def get_nba_teams():
    """
    Gets NBA teams from TheSportsDB.
    """

    print("\n====================================")
    print("STEP 2: DOWNLOADING NBA TEAMS")
    print("====================================")

    data = make_request(
        "search_all_teams.php",
        {
            "l": "NBA"
        }
    )

    if data is None:
        return pd.DataFrame()

    teams = data.get("teams")

    if not teams:
        print("No NBA teams returned.")
        return pd.DataFrame()

    team_rows = []

    for team in teams:

        row = {
            "team_id": team.get("idTeam"),
            "team_name": team.get("strTeam"),
            "alternate_name": team.get("strTeamAlternate"),
            "league": team.get("strLeague"),
            "sport": team.get("strSport"),
            "country": team.get("strCountry"),
            "stadium": team.get("strStadium"),
            "location": team.get("strLocation"),
            "year_formed": team.get("intFormedYear")
        }

        team_rows.append(row)

    df = pd.DataFrame(team_rows)

    print("\nNBA teams downloaded:")
    print(df[["team_name", "location"]].head(10))

    return df


# ============================================================
# 3. GET NBA SEASON GAMES
# ============================================================

def get_nba_games(league_id):
    """
    Downloads NBA games for the selected season.
    """

    print("\n====================================")
    print("STEP 3: DOWNLOADING NBA GAMES")
    print("====================================")

    data = make_request(
        "eventsseason.php",
        {
            "id": league_id,
            "s": SEASON
        }
    )

    if data is None:
        return pd.DataFrame()

    events = data.get("events")

    if not events:
        print(f"No games found for season {SEASON}")
        return pd.DataFrame()

    game_rows = []

    for game in events:

        row = {
            "event_id": game.get("idEvent"),

            "date": game.get("dateEvent"),

            "time": game.get("strTime"),

            "season": game.get("strSeason"),

            "home_team": game.get("strHomeTeam"),

            "away_team": game.get("strAwayTeam"),

            "home_team_id": game.get("idHomeTeam"),

            "away_team_id": game.get("idAwayTeam"),

            "home_score": game.get("intHomeScore"),

            "away_score": game.get("intAwayScore"),

            "venue": game.get("strVenue"),

            "status": game.get("strStatus"),

            "league": game.get("strLeague")
        }

        game_rows.append(row)

    df = pd.DataFrame(game_rows)

    print("\nGames downloaded:")
    print(
        df[
            [
                "date",
                "home_team",
                "away_team",
                "home_score",
                "away_score"
            ]
        ].head(10)
    )

    return df


# ============================================================
# 4. CREATE SIMPLE FEATURES
# ============================================================

def create_features(games_df):
    """
    Creates several useful features from game data.
    """

    print("\n====================================")
    print("STEP 4: CREATING FEATURES")
    print("====================================")

    if games_df.empty:
        print("No game data available.")
        return games_df

    df = games_df.copy()

    # Convert scores into numeric values
    df["home_score"] = pd.to_numeric(
        df["home_score"],
        errors="coerce"
    )

    df["away_score"] = pd.to_numeric(
        df["away_score"],
        errors="coerce"
    )

    # ---------------------------------------------------------
    # TOTAL POINTS
    # ---------------------------------------------------------

    df["total_points"] = (
        df["home_score"]
        +
        df["away_score"]
    )

    # ---------------------------------------------------------
    # SCORE DIFFERENCE
    # ---------------------------------------------------------

    df["score_difference"] = (
        df["home_score"]
        -
        df["away_score"]
    )

    # ---------------------------------------------------------
    # HOME TEAM WIN
    # ---------------------------------------------------------

    df["home_win"] = (
        df["home_score"]
        >
        df["away_score"]
    ).astype("Int64")

    print("\nExample engineered features:")

    columns = [
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "total_points",
        "score_difference",
        "home_win"
    ]

    print(df[columns].head(10))

    return df


# ============================================================
# 5. SAVE THE DATA
# ============================================================

def save_data(teams_df, games_df):

    print("\n====================================")
    print("STEP 5: SAVING DATA")
    print("====================================")

    if not teams_df.empty:

        teams_df.to_csv(
            "nba_teams.csv",
            index=False
        )

        print("Saved: nba_teams.csv")

    if not games_df.empty:

        games_df.to_csv(
            "nba_games.csv",
            index=False
        )

        print("Saved: nba_games.csv")


# ============================================================
# MAIN PROGRAM
# ============================================================

def main():

    print("\n")
    print("==========================================")
    print(" NBA DATA INGESTION - THESPORTSDB")
    print("==========================================")

    print("\nData Source: TheSportsDB")
    print("API Version: V1")
    print("Free API Key: 123")
    print("Season:", SEASON)

    # --------------------------------------------------------
    # Find NBA league
    # --------------------------------------------------------

    nba_league = find_nba_league()

    if nba_league is None:

        print("\nNBA league could not be found.")
        print("Program stopped.")

        return

    league_id = nba_league["idLeague"]

    print("\nUsing NBA League ID:", league_id)

    # Small delay so we don't hit API too quickly
    time.sleep(1)

    # --------------------------------------------------------
    # Download teams
    # --------------------------------------------------------

    teams_df = get_nba_teams()

    time.sleep(1)

    # --------------------------------------------------------
    # Download games
    # --------------------------------------------------------

    games_df = get_nba_games(league_id)

    # --------------------------------------------------------
    # Feature engineering
    # --------------------------------------------------------

    games_df = create_features(games_df)

    # --------------------------------------------------------
    # Save everything
    # --------------------------------------------------------

    save_data(
        teams_df,
        games_df
    )

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    print("\n====================================")
    print("DATA INGESTION COMPLETE")
    print("====================================")

    print("\nNumber of teams:", len(teams_df))
    print("Number of games:", len(games_df))

    print("\nFiles created:")

    print("nba_teams.csv")
    print("nba_games.csv")

    print("\nDone!")


# ============================================================
# START PROGRAM
# ============================================================

if __name__ == "__main__":
    main()



 NBA DATA INGESTION - THESPORTSDB

Data Source: TheSportsDB
API Version: V1
Free API Key: 123
Season: 2025-2026

STEP 1: FINDING NBA LEAGUE

Requesting: https://www.thesportsdb.com/api/v1/json/123/search_all_leagues.php
Found league: Basketball All-Star Games
Found league: FIBA AmeriCup
Found league: NBA

NBA FOUND!
League ID: 4387

Using NBA League ID: 4387

STEP 2: DOWNLOADING NBA TEAMS

Requesting: https://www.thesportsdb.com/api/v1/json/123/search_all_teams.php

NBA teams downloaded:
               team_name                        location
0          Atlanta Hawks                Atlanta, Georgia
1         Boston Celtics           Boston, Massachusetts
2          Brooklyn Nets              Brooklyn, New York
3      Charlotte Hornets       Charlotte, North Carolina
4          Chicago Bulls               Chicago, Illinois
5    Cleveland Cavaliers                 Cleveland, Ohio
6       Dallas Mavericks                   Dallas, Texas
7         Denver Nuggets                Denver, C

In [2]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles in folder:")
print(os.listdir())

Current folder:
/Users/teahun/Documents/DTSC Group Project

Files in folder:
['nba_games.csv', 'nba_data_ingestion.ipynb', 'nba_teams.csv']


In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error


# ============================================================
# 1. LOAD OUR STORED NBA DATA
# ============================================================

print("\n==============================")
print("LOADING NBA DATA")
print("==============================")

df = pd.read_csv("nba_games.csv")

print("\nFirst rows:")
print(df.head())

print("\nNumber of games:", len(df))


# ============================================================
# 2. REMOVE GAMES WITHOUT SCORES
# ============================================================

df["home_score"] = pd.to_numeric(
    df["home_score"],
    errors="coerce"
)

df["away_score"] = pd.to_numeric(
    df["away_score"],
    errors="coerce"
)

df = df.dropna(
    subset=["home_score", "away_score"]
)


# ============================================================
# 3. CREATE TARGET VARIABLE
# ============================================================

# What we want the model to predict:
# total points scored in the game

df["total_points"] = (
    df["home_score"]
    +
    df["away_score"]
)


# ============================================================
# 4. SELECT FEATURES
# ============================================================

# X = information given TO the model

X = df[
    [
        "home_team",
        "away_team"
    ]
]


# y = what we want the model to predict

y = df["total_points"]


print("\n==============================")
print("FEATURES")
print("==============================")

print(X.head())


print("\n==============================")
print("TARGET")
print("==============================")

print(y.head())


# ============================================================
# 5. SPLIT TRAINING AND TEST DATA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)


print("\nTraining games:", len(X_train))
print("Testing games:", len(X_test))


# ============================================================
# 6. PREPROCESS TEAM NAMES
# ============================================================

# Machine-learning models cannot directly understand
# strings like "Boston Celtics".
#
# OneHotEncoder converts team names into numbers.

categorical_features = [
    "home_team",
    "away_team"
]


preprocessor = ColumnTransformer(
    transformers=[
        (
            "teams",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)


# ============================================================
# 7. CREATE REGRESSION MODEL
# ============================================================

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "regression",
            LinearRegression()
        )
    ]
)


# ============================================================
# 8. TRAIN MODEL
# ============================================================

print("\n==============================")
print("TRAINING MODEL")
print("==============================")

model.fit(
    X_train,
    y_train
)

print("Model training complete!")


# ============================================================
# 9. TEST MODEL
# ============================================================

predictions = model.predict(X_test)


print("\n==============================")
print("MODEL TEST RESULTS")
print("==============================")

for actual, predicted in zip(
    y_test,
    predictions
):

    print(
        f"Actual: {actual:.0f} | "
        f"Predicted: {predicted:.1f}"
    )


mae = mean_absolute_error(
    y_test,
    predictions
)


print(
    "\nMean Absolute Error:",
    round(mae, 2)
)


# ============================================================
# 10. MAKE A NEW PREDICTION
# ============================================================

example_game = pd.DataFrame(
    {
        "home_team": [
            "Boston Celtics"
        ],

        "away_team": [
            "Los Angeles Lakers"
        ]
    }
)


predicted_total = model.predict(
    example_game
)[0]


print("\n==============================")
print("NEW GAME PREDICTION")
print("==============================")

print(
    "Boston Celtics vs Los Angeles Lakers"
)

print(
    "Predicted Total Points:",
    round(predicted_total, 1)
)


LOADING NBA DATA

First rows:
   event_id        date      time     season           home_team  \
0   2292728  2025-10-02  16:00:00  2025-2026  Philadelphia 76ers   
1   2292729  2025-10-04  02:00:00  2025-2026  Los Angeles Lakers   
2   2292730  2025-10-04  15:00:00  2025-2026  Philadelphia 76ers   
3   2357311  2025-10-05  00:00:00  2025-2026          Miami Heat   
4   2357312  2025-10-05  01:00:00  2025-2026      Denver Nuggets   

                away_team  home_team_id  away_team_id  home_score  away_score  \
0         New York Knicks        134863        134862          99          84   
1            Phoenix Suns        134867        134868          81         103   
2         New York Knicks        134863        134862         104         113   
3           Orlando Magic        134882        134883         118         126   
4  Minnesota Timberwolves        134885        134886         116         126   

            venue status league  total_points  score_difference  home_win